In [1]:
from selenium import webdriver as wb
# 컴퓨터용 키보드(enter, del, end 등) 컴퓨터에게 키보드를 전달하는 역할
from selenium.webdriver.common.keys import Keys
# 선택자를 구분할 때 사용하는 라이브러리(필수)
from selenium.webdriver.common.by import By
import time
import re
import pandas as pd
from tqdm import tqdm
from datetime import datetime

In [23]:
from urllib.parse import quote
keywords = "육아+불편" # 힘들 OR 불안 OR 걱정 OR 어렵 OR 불편
url=f'https://brunch.co.kr/search?q={keywords}&type=article'
driver=wb.Chrome()
driver.get(url)

In [24]:
# 스크롤 끝까지 내리기(무한 스크롤/추가 로딩 대응)
pause_sec = 1.2
max_no_change = 3  # 높이 변화가 N번 연속 없으면 종료

no_change_count = 0
last_height = driver.execute_script("return document.body.scrollHeight")

while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(pause_sec)

    new_height = driver.execute_script("return document.body.scrollHeight")

    if new_height == last_height:
        no_change_count += 1
        if no_change_count >= max_no_change:
            break
    else:
        no_change_count = 0
        last_height = new_height

print("스크롤 완료")
# driver.quit()

스크롤 완료


In [25]:
href_list = []

a_tags = driver.find_elements(By.CSS_SELECTOR, "a.link_post")
for a in tqdm(a_tags, desc="href 수집중"):
    href = a.get_attribute("href")
    if href:
        href_list.append(href)

# (선택) 중복 제거(순서 유지)
seen = set()
href_list = [h for h in href_list if not (h in seen or seen.add(h))]

print("수집 개수:", len(href_list))
print("샘플:", href_list[:5])

href 수집중: 100%|██████████████████████████████████████████████████████████████████| 703/703 [00:03<00:00, 181.73it/s]

수집 개수: 703
샘플: ['https://brunch.co.kr/@@f9Dp/99', 'https://brunch.co.kr/@@dVh7/213', 'https://brunch.co.kr/@@cnII/127', 'https://brunch.co.kr/@@9xIX/40', 'https://brunch.co.kr/@@heFj/44']


In [26]:
len(href_list)

703

In [27]:
naver_list = []
tistory_list = []
brunch_list = []
reddit_list = []

for url in href_list:
    if "blog.naver.com" in url or "m.blog.naver.com" in url:
        naver_list.append(url)

    elif "tistory.com" in url:
        tistory_list.append(url)

    elif "brunch.co.kr" in url:
        brunch_list.append(url)

    elif "reddit.com" in url:
        reddit_list.append(url)

print("네이버:", naver_list)
print("티스토리:", tistory_list)
print("브런치:", brunch_list)
print("레딧:", reddit_list)


네이버: []
티스토리: []
브런치: ['https://brunch.co.kr/@@f9Dp/99', 'https://brunch.co.kr/@@dVh7/213', 'https://brunch.co.kr/@@cnII/127', 'https://brunch.co.kr/@@9xIX/40', 'https://brunch.co.kr/@@heFj/44', 'https://brunch.co.kr/@@8zGO/217', 'https://brunch.co.kr/@@cNck/373', 'https://brunch.co.kr/@@H3s/591', 'https://brunch.co.kr/@@91x/129', 'https://brunch.co.kr/@@cNck/369', 'https://brunch.co.kr/@@hDn0/31', 'https://brunch.co.kr/@@dyMr/51', 'https://brunch.co.kr/@@2Sgv/27', 'https://brunch.co.kr/@@7XZu/40', 'https://brunch.co.kr/@@zIH/6790', 'https://brunch.co.kr/@@h1Pi/275', 'https://brunch.co.kr/@@hD3v/7', 'https://brunch.co.kr/@@4qH0/40', 'https://brunch.co.kr/@@fnGv/39', 'https://brunch.co.kr/@@imFl/45', 'https://brunch.co.kr/@@cCMB/79', 'https://brunch.co.kr/@@ctkm/184', 'https://brunch.co.kr/@@3hX/41', 'https://brunch.co.kr/@@5hIX/68', 'https://brunch.co.kr/@@bVTB/36', 'https://brunch.co.kr/@@gGaE/25', 'https://brunch.co.kr/@@TAx/10', 'https://brunch.co.kr/@@dkY2/351', 'https://brunch.co.

In [28]:
print(len(naver_list))
print(len(tistory_list))
print(len(brunch_list))
print(len(reddit_list))

0
0
703
0


### 브런치 크롤링 코드

In [29]:
import re
import csv
from tqdm import tqdm
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import TimeoutException

driver = wb.Chrome()
wait = WebDriverWait(driver, 15)

import re

def brunch_date_to_yyyy_mm_dd(s: str) -> str:
    """
    'Mar 24. 2025' -> '2025-03-24'
    로케일 영향 없이 변환.
    실패 시 '' 반환
    """
    if not s:
        return ""

    s = s.strip()
    # 점/콤마 등을 공백으로 정리
    s = re.sub(r"[,\.\u00B7]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    # 예: "Mar 24 2025"
    m = re.match(r"([A-Za-z]{3})\s+(\d{1,2})\s+(\d{4})$", s)
    if not m:
        return ""

    mon_str, day_str, year_str = m.group(1), m.group(2), m.group(3)

    month_map = {
        "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
        "May": 5, "Jun": 6, "Jul": 7, "Aug": 8,
        "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
    }

    mon = month_map.get(mon_str.title())
    if not mon:
        return ""

    day = int(day_str)
    year = int(year_str)

    # 간단 유효성 체크(추측입니다: 여기까지 오면 대부분 정상)
    if not (1 <= day <= 31):
        return ""

    return f"{year:04d}-{mon:02d}-{day:02d}"

def extract_brunch_date(driver) -> str:
    """
    브런치 작성일 추출: 'Mar 24. 2025' -> '2025-03-24'
    실패 시 '' 반환
    """
    # 1) 화면에 보이는 날짜(span.f_l.date)
    try:
        els = driver.find_elements(By.CSS_SELECTOR, "span.f_l.date")
        if els:
            raw = (els[0].text or "").strip()
            conv = brunch_date_to_yyyy_mm_dd(raw)
            if conv:
                return conv
    except:
        pass

    # 2) fallback: og(또는 meta) 쪽에 date가 있는 경우 대비 (있으면 사용)
    # 브런치는 글마다 메타 구성이 다를 수 있어서 "있으면"만 사용
    for sel in [
        'meta[property="article:published_time"]',
        'meta[name="article:published_time"]',
        'meta[property="og:article:published_time"]'
    ]:
        metas = driver.find_elements(By.CSS_SELECTOR, sel)
        if metas:
            v = (metas[0].get_attribute("content") or "").strip()
            # 예: 2025-03-24T... 이런 형식일 수 있으니 앞 10자리만
            if len(v) >= 10 and re.match(r"\d{4}-\d{2}-\d{2}", v[:10]):
                return v[:10]

    return ""

def extract_brunch_title(driver) -> str:
    # 1) og:title (가장 안정적)
    metas = driver.find_elements(By.CSS_SELECTOR, 'meta[property="og:title"]')
    if metas:
        t = (metas[0].get_attribute("content") or "").strip()
        if t:
            return t

    # 2) 너무 긴 '>' 체인 대신, 핵심 구조만 느슨하게 매칭
    candidates = [
        "div.article_contents div.wrap_cover h1 > span",
        "div.article_contents div.wrap_cover h1 span",
        "div.wrap_cover h1 > span",
        "div.wrap_cover h1 span",
    ]

    for css in candidates:
        try:
            el = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, css))
            )
            t = (el.text or "").strip()
            if t:
                return t
            tc = (driver.execute_script("return arguments[0].textContent;", el) or "").strip()
            if tc:
                return tc
        except TimeoutException:
            continue

    # 3) 마지막 fallback: h1 자체에서 텍스트 추출
    for css in ["div.article_contents h1", "h1"]:
        els = driver.find_elements(By.CSS_SELECTOR, css)
        for el in els:
            t = (el.text or "").strip()
            if t:
                return t
            tc = (driver.execute_script("return arguments[0].textContent;", el) or "").strip()
            if tc:
                return tc

    return ""

from selenium.webdriver.common.by import By

def extract_brunch_content(driver) -> str:
    """
    브런치 '본문만' 수집:
    - 제목(h1 등), 날짜(span.f_l.date), 상단 메타/공유 영역 텍스트가 섞이지 않게
    - 본문 컨테이너 내부에서 본문에 해당하는 태그만 모아 합침
    """

    # 1) 본문이 들어있는 컨테이너 후보 (큰 틀)
    containers = [
        "div.wrap_view_article",
        "div#wrapArticle",
        "div.wrap_article",
        "article",
        "main"
    ]

    root = None
    for csel in containers:
        els = driver.find_elements(By.CSS_SELECTOR, csel)
        if els:
            root = els[0]
            break

    # 컨테이너를 못 찾으면 최후 fallback: body 전체
    if root is None:
        body = driver.find_elements(By.TAG_NAME, "body")
        return (body[0].text or "").strip() if body else ""

    # 2) 컨테이너 내부에서 "본문 태그들"만 수집
    #    (제목/날짜는 보통 h1 / span.f_l.date 형태이므로 여기 포함 안 시킴)
    content_selectors = [
        "p",
        "b",
        "li",
        "blockquote",
        "span"  # 글에 코드가 있을 수 있으면 유지
    ]

    texts = []
    seen = set()

    for sel in content_selectors:
        for e in root.find_elements(By.CSS_SELECTOR, sel):
            t = (e.text or "").strip()
            if not t:
                continue

            # 3) 제목/날짜/메타성 텍스트를 한 번 더 방어적으로 걸러냄
            #    (페이지마다 구조가 달라 h2/h3에 제목이 들어가는 경우가 아주 드물게 있을 수 있음)
            if re.search(r"^[A-Za-z]{3}\s+\d{1,2}[.,]?\s+\d{4}$", t):   # 예: Mar 24. 2025
                continue

            # 중복 제거(순서 유지)
            if t not in seen:
                seen.add(t)
                texts.append(t)

    # 4) 본문이 너무 짧으면(실패 가능성) root.text 대신 body.text fallback
    merged = "\n".join(texts).strip()
    if len(merged) >= 30:
        return merged

    body = driver.find_elements(By.TAG_NAME, "body")
    return (body[0].text or "").strip() if body else merged

with open(f"블로그_브런치_{keywords}({len(url)}건).csv", "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["출처", "제목", "내용", "작성일", "링크"])

    for url in tqdm(brunch_list):
        try:
            driver.get(url)

            # 페이지에 og:title이나 h1 중 하나 뜰 때까지 대기(없어도 진행)
            try:
                wait.until(lambda d: d.find_elements(By.CSS_SELECTOR, 'meta[property="og:title"]') or
                                    d.find_elements(By.CSS_SELECTOR, "h1") or
                                    d.find_elements(By.CSS_SELECTOR, "span.f_l.date"))
            except TimeoutException:
                pass
            date = extract_brunch_date(driver)
            title = extract_brunch_title(driver)
            content = extract_brunch_content(driver)

            writer.writerow(["블로그_브런치", title, content, date, url])

        except Exception as e:
            writer.writerow(["블로그_브런치", "", "", "", url])
            print("ERROR:", url, type(e).__name__)
            continue


100%|████████████████████████████████████████████████████████████████████████████████| 703/703 [22:25<00:00,  1.91s/it]
